In [1]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress
from sqlalchemy import create_engine, event, text
from sqlalchemy.engine import URL

ROOT = r'D:\Bluestock work\bluestock-fintech-analysis1'
RAW_DIR = os.path.join(ROOT, 'data', 'raw')
PROCESSED_DIR = os.path.join(ROOT, 'data', 'processed')
REPORTS_DIR = os.path.join(ROOT, 'reports')
DB_PATH = os.path.join(PROCESSED_DIR, 'bluestock_dw.db')

TRADING_DAYS = 252
RF_ANNUAL = 0.065
RF_DAILY = RF_ANNUAL / TRADING_DAYS

engine = create_engine(URL.create('sqlite', database=DB_PATH))

@event.listens_for(engine, 'connect')
def turn_on_foreign_keys(dbapi_connection, connection_record):
    cursor = dbapi_connection.cursor()
    cursor.execute('PRAGMA foreign_keys = ON')
    cursor.close()

In [2]:
bench_path = glob.glob(os.path.join(RAW_DIR, '*10_benchmark_indices*'))[0]
bench = pd.read_csv(bench_path)

print(bench.shape)
print(bench.dtypes)
print(bench['date'].head())
print(bench['index_name'].value_counts())

(8050, 3)
date               str
index_name         str
close_value    float64
dtype: object
0    2022-01-03
1    2022-01-04
2    2022-01-05
3    2022-01-06
4    2022-01-07
Name: date, dtype: str
index_name
NIFTY50            1150
NIFTY100           1150
NIFTY_MIDCAP150    1150
BSE_SMALLCAP       1150
NIFTY500           1150
CRISIL_LIQUID      1150
CRISIL_GILT        1150
Name: count, dtype: int64


In [3]:
NIFTY_NAME = 'Nifty 100'
DATE_FORMAT = '%Y-%m-%d'

idx = bench[bench['index_name'] == NIFTY_NAME].copy()
print('Rows for', NIFTY_NAME, ':', len(idx))

idx['date'] = pd.to_datetime(idx['date'], format=DATE_FORMAT, errors='coerce')
idx['close_value'] = pd.to_numeric(idx['close_value'], errors='coerce')

print('Dates not readable:', idx['date'].isnull().sum())
print('Close not a number:', idx['close_value'].isnull().sum())
print('Close zero or below:', (idx['close_value'] <= 0).sum())
print('Repeated dates:', idx.duplicated(subset=['date']).sum())

idx = idx.dropna(subset=['date', 'close_value'])
idx = idx[idx['close_value'] > 0]
idx = idx.drop_duplicates(subset=['date'], keep='last').sort_values('date')

print(idx['date'].min().date(), 'to', idx['date'].max().date(), '| rows:', len(idx))

Rows for Nifty 100 : 0
Dates not readable: 0
Close not a number: 0
Close zero or below: 0
Repeated dates: 0
NaT to NaT | rows: 0


In [ ]:
idx['date'] = pd.to_datetime(idx['date'])

idx_close = idx.set_index('date')['close_value']
idx_daily = idx_close.resample('D').last().ffill(limit=7)

idx_daily.index = pd.to_datetime(idx_daily.index)


print('Index days:', len(idx_close), '| calendar days:', len(idx_daily))
print('Calendar days still empty:', idx_daily.isnull().sum())

<class 'pandas.api.typing.NaTType'>
Index days: 0 | calendar days: 0
Calendar days still empty: 0


In [14]:
print(type(idx_daily.index))  # <class 'pandas.core.indexes.datetimes.DatetimeIndex'>
print(type(idx_daily.first_valid_index()))  # <class 'pandas._libs.tslibs.timestamps.Timestamp'>

<class 'pandas.DatetimeIndex'>
<class 'NoneType'>


In [7]:
returns = pd.read_sql_query("""
SELECT r.amfi_code, d.full_date, r.daily_return, r.days_since_prev
FROM fact_daily_return AS r
JOIN dim_date AS d ON d.date_key = r.date_key
ORDER BY r.amfi_code, r.date_key
""", engine)

returns['date'] = pd.to_datetime(returns['full_date'])
returns['prev_date'] = returns['date'] - pd.to_timedelta(returns['days_since_prev'], unit='D')

close_end = idx_daily.reindex(returns['date']).to_numpy()
close_start = idx_daily.reindex(returns['prev_date']).to_numpy()
returns['index_return'] = close_end / close_start - 1

print('Return rows:', len(returns))
print('Rows with no index return:', returns['index_return'].isnull().sum())
print('Correlation of fund and index returns:', round(returns[['daily_return', 'index_return']].corr().iloc[0, 1], 2))

Return rows: 45960
Rows with no index return: 45960
Correlation of fund and index returns: nan


In [18]:
first_date = returns.groupby('amfi_code')['date'].min()
COMMON_START = first_date.max()

WINDOW_START = max(COMMON_START, idx_daily.first_valid_index())
WINDOW_END = min(returns['date'].max(), idx_daily.last_valid_index())

reg = returns[(returns['date'] >= WINDOW_START) & (returns['date'] <= WINDOW_END)]
reg = reg.dropna(subset=['index_return'])

years = (WINDOW_END - WINDOW_START).days / 365.25
print('Window:', WINDOW_START.date(), 'to', WINDOW_END.date(), '| years:', round(years, 2))
print('Funds:', reg['amfi_code'].nunique(), '| rows:', len(reg))
print(reg.groupby('amfi_code').size().describe().round(0))

TypeError: '>' not supported between instances of 'NoneType' and 'Timestamp'

In [19]:
print('Index names in the file (in quotes, to show hidden spaces):')
for name in bench['index_name'].dropna().unique():
    print('  ', repr(name))

print()
print('Name used:', repr(NIFTY_NAME))
print('Rows with this exact name:', (bench['index_name'] == NIFTY_NAME).sum())
print('Rows with "100" in the name:', bench['index_name'].str.contains('100', na=False).sum())

part = bench[bench['index_name'] == NIFTY_NAME]

print()
print('Raw dates (first 5):', part['date'].head().tolist())
print('Date format used:', DATE_FORMAT)
parsed = pd.to_datetime(part['date'], format=DATE_FORMAT, errors='coerce')
print('Dates that could be read:', parsed.notnull().sum(), 'of', len(part))

closes = pd.to_numeric(part['close_value'], errors='coerce')
print('Close values that are numbers:', closes.notnull().sum(), 'of', len(part))

Index names in the file (in quotes, to show hidden spaces):
   'NIFTY50'
   'NIFTY100'
   'NIFTY_MIDCAP150'
   'BSE_SMALLCAP'
   'NIFTY500'
   'CRISIL_LIQUID'
   'CRISIL_GILT'

Name used: 'Nifty 100'
Rows with this exact name: 0
Rows with "100" in the name: 1150

Raw dates (first 5): []
Date format used: %Y-%m-%d
Dates that could be read: 0 of 0
Close values that are numbers: 0 of 0


In [20]:
NIFTY_NAME = 'Nifty 100'

In [21]:
idx = bench[bench['index_name'] == NIFTY_NAME].copy()
assert len(idx) > 0, 'No rows for this index name. Check NIFTY_NAME.'

idx['date'] = pd.to_datetime(idx['date'], format=DATE_FORMAT, errors='coerce')
idx['close_value'] = pd.to_numeric(
    idx['close_value'].astype(str).str.replace(',', '', regex=False), errors='coerce')

print('Dates not readable:', idx['date'].isnull().sum())
print('Close not a number:', idx['close_value'].isnull().sum())
print('Close zero or below:', (idx['close_value'] <= 0).sum())
print('Repeated dates:', idx.duplicated(subset=['date']).sum())

idx = idx.dropna(subset=['date', 'close_value'])
idx = idx[idx['close_value'] > 0]
idx = idx.drop_duplicates(subset=['date'], keep='last').sort_values('date')
assert len(idx) > 0, 'All rows were removed. Check DATE_FORMAT and the close values.'

print(idx['date'].min().date(), 'to', idx['date'].max().date(), '| rows:', len(idx))

idx_close = idx.set_index('date')['close_value']
idx_daily = idx_close.resample('D').last().ffill(limit=7)
assert idx_daily.notnull().any(), 'The calendar series is empty.'

print('Index days:', len(idx_close), '| calendar days:', len(idx_daily))
print('Calendar days still empty:', idx_daily.isnull().sum())

AssertionError: No rows for this index name. Check NIFTY_NAME.